# Zero-shot IAB classification \u2014 single-stage `Qwen3-Embedding-4B`, Tier 1\u20132, tuned for Colab T4

## What this notebook does (in plain terms)

Tag the (already cleaned) text of a web page with one or more **IAB Content Taxonomy** categories (Tier 1 and Tier 2 only), with a relevance score and a filter, zero-shot and multilingual. Compared to the reranking notebook, this one is **single-stage**: no cross-encoder. We embed the page and every category and rank by similarity \u2014 simpler and faster \u2014 but we offset the lack of a reranker by using a **stronger, instruction-aware embedding model**, `Qwen/Qwen3-Embedding-4B`.

Because there's only one model in memory (no reranker), the 4B encoder fits a free T4 comfortably when loaded in half precision.

## The key detail: Qwen3-Embedding is *instruction-aware* and *asymmetric*

This is the part that's easy to get wrong, so it's handled explicitly here. Qwen3-Embedding distinguishes two roles:

- the **query** \u2014 encoded together with a short natural-language **instruction** describing the task;
- the **documents** \u2014 the corpus you search over, encoded **plain**, with no instruction.

For our task the **page is the query** (the thing whose intent we express) and the **IAB labels are the documents** (the fixed corpus we pre-embed). So:

- labels are embedded as-is (`Sports`, `Business and Finance > Personal Finance`, \u2026);
- each page is embedded with a task instruction prepended, in Qwen3's required format:

  ```
  Instruct: {task description}
  Query: {page text}
  ```

A task-specific instruction is what unlocks Qwen3's quality (the model is trained to follow it, typically worth a few points). In `sentence-transformers` we apply it simply by passing `prompt=` to `model.encode(...)` for pages, and nothing for labels. Last-token pooling and left padding (which Qwen3 needs) are handled by the `sentence-transformers` integration, so we don't touch them.

## Model and T4 fit

- **`Qwen/Qwen3-Embedding-4B`**, loaded in **fp16** \u2192 ~8 GB of weights, well within the T4's 16 GB once there's no second model to host. We also cap `max_seq_length` to keep memory and latency in check (our excerpts are short; raise it for long raw pages \u2014 Qwen3 supports up to 32k tokens).
- If you ever hit memory limits, drop to `Qwen/Qwen3-Embedding-0.6B` (one line) \u2014 it keeps the same instruction interface.

> **Run on Colab:** `Runtime \u2192 Change runtime type \u2192 GPU (T4)`, then run top to bottom. The first run downloads the model (~8 GB in fp16) and the taxonomy from GitHub. Without a reranker, thresholds act on the **cosine** scale again (not sigmoid).

### Step 0 \u2014 Dependencies

Qwen3-Embedding needs a recent `sentence-transformers` and `transformers` (\u2265 4.51). We upgrade only those two \u2014 **not** numpy/pandas, whose mid-session upgrade breaks the Colab kernel.

In [1]:
# 0. Dependencies (Qwen3-Embedding needs recent sentence-transformers + transformers;
#    do NOT force-upgrade numpy/pandas: it breaks the running Colab kernel)
!pip install -q -U 'sentence-transformers>=2.7.0' 'transformers>=4.51.0'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 119.8 MB/s eta 0:00:00


### Step 1 \u2014 Load the IAB taxonomy, restricted to Tier 1 + Tier 2

Download the official 3.1 TSV and keep only nodes with a tier path of length \u2264 2 (~360 categories). Each node keeps its id, parent, tier path and an **enriched label** (`Tier 1 > Tier 2`, or `Tier 1`); we also build the Tier 1 \u2192 Tier 2 children map for the upward aggregation in Step 3.

In [2]:
import io, requests, numpy as np, pandas as pd

RAW_URL = ("https://raw.githubusercontent.com/InteractiveAdvertisingBureau/"
           "Taxonomies/main/Content%20Taxonomies/Content%20Taxonomy%203.1.tsv")

txt = requests.get(RAW_URL, timeout=60); txt.raise_for_status()
lines = txt.text.splitlines()
hdr = next(i for i, l in enumerate(lines) if l.split('\t')[0].strip() == 'Unique ID')   # skip title rows
tax = pd.read_csv(io.StringIO("\n".join(lines[hdr:])), sep='\t', dtype=str).fillna('')
tax.columns = [c.strip() for c in tax.columns]

MAX_TIER  = 2                                  # <-- keep only Tier 1 and Tier 2
TIER_COLS = ['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4']
nodes = []
for _, r in tax.iterrows():
    uid = str(r.get('Unique ID', '')).strip()
    if not uid:
        continue
    path = [str(r[c]).strip() for c in TIER_COLS if c in tax.columns and str(r[c]).strip()]
    if not path or len(path) > MAX_TIER:       # drop Tier 3 / Tier 4
        continue
    nodes.append({
        'id': uid,
        'parent': str(r.get('Parent', '')).strip(),
        'name': str(r.get('Name', '')).strip() or path[-1],
        'path': path,
        'tier': len(path),
        'enriched': ' > '.join(path),
    })

id2idx   = {n['id']: i for i, n in enumerate(nodes)}
children = {n['id']: [] for n in nodes}
for n in nodes:
    if n['parent'] in children:
        children[n['parent']].append(n['id'])
label_texts = [n['enriched'] for n in nodes]

print(f"Nodes kept (Tier 1+2): {len(nodes)}")
print("Per tier:", pd.Series([n['tier'] for n in nodes]).value_counts().sort_index().to_dict())
print("Examples:", label_texts[:6])

Nodes kept (Tier 1+2): 360
Per tier: {1: 37, 2: 323}
Examples: ['Attractions', 'Attractions > Amusement and Theme Parks', 'Attractions > Bars & Restaurants', 'Attractions > Casinos & Gambling', 'Attractions > Historic Site and Landmark Tours', 'Attractions > Malls & Shopping Centers']


### Step 2 \u2014 Load Qwen3-Embedding-4B and embed the labels

We load the 4B model in **fp16** on GPU (fp32 on CPU fallback) and cap `max_seq_length`. The **labels are embedded plain** (they are the documents). The **task instruction** is defined here and applied later only to pages (the query side). `normalize_embeddings=True` makes the dot product equal to cosine similarity.

In [3]:
from sentence_transformers import SentenceTransformer
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", DEVICE)

MODEL_NAME = 'Qwen/Qwen3-Embedding-4B'        # fallback if VRAM is tight: 'Qwen/Qwen3-Embedding-0.6B'
model_kwargs = {'torch_dtype': torch.float16} if DEVICE == 'cuda' else {}   # fp16 -> ~8 GB, fits a T4
model = SentenceTransformer(MODEL_NAME, device=DEVICE, model_kwargs=model_kwargs)
model.max_seq_length = 1024                   # short excerpts; raise for long raw pages (Qwen3 supports 32k)

# Instruction goes on the QUERY (page) side only; labels (documents) are encoded plain.
INSTRUCT = ("Instruct: Given the text content of a web page, retrieve the IAB content "
            "taxonomy categories that best describe what the page is about.\nQuery: ")

# Documents = IAB labels, encoded WITHOUT any instruction
label_emb = model.encode(label_texts, batch_size=16, normalize_embeddings=True,
                         show_progress_bar=True, convert_to_numpy=True)
print("Label matrix:", label_emb.shape)

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Label matrix: (360, 2560)


### Step 3 \u2014 The single-stage classifier

For each page:

1. **Encode the page as a query** \u2014 `model.encode([text], prompt=INSTRUCT)` prepends the instruction in Qwen3's format. This is the one place the query/document asymmetry matters.
2. **Score** \u2014 cosine vs all Tier 1\u20132 labels (no reranker).
3. **Aggregate up** \u2014 each Tier 1 inherits the **best cosine of its Tier 2 children**, so a broad bucket surfaces even when only a specific Tier 2 matched.
4. **Filter & granularity** \u2014 keep nodes with `score \u2265 floor` and within `margin` of the page's best, cap at `top_k`; `'specific'` drops a Tier 1 already covered by a selected Tier 2, `'tier1'` collapses to broad buckets, `'all'` keeps both.

Scores are on the **cosine** scale, so `floor`/`margin` are different from the reranker notebook \u2014 tune them on the 100 examples below.

In [4]:
def classify(text, floor=0.40, margin=0.10, top_k=4, granularity='specific'):
    # 1. page = query (instruction prepended); 2. cosine vs plain label embeddings
    q = model.encode([text], prompt=INSTRUCT, normalize_embeddings=True, convert_to_numpy=True)[0]
    raw = (label_emb @ q).astype(float)

    # 3. each Tier 1 inherits the best cosine among its Tier 2 children
    agg = raw.copy()
    for i, n in enumerate(nodes):
        if n['tier'] == 2 and n['parent'] in id2idx:
            pi = id2idx[n['parent']]
            if raw[i] > agg[pi]:
                agg[pi] = raw[i]

    # 4. filter (absolute floor + relative margin) then top_k
    order = np.argsort(-agg)
    top = float(agg[order[0]])
    sel = [int(i) for i in order if agg[i] >= floor and agg[i] >= top - margin]

    if granularity == 'specific':       # drop a Tier 1 already covered by a selected Tier 2
        covered = {id2idx[nodes[i]['parent']] for i in sel
                   if nodes[i]['tier'] == 2 and nodes[i]['parent'] in id2idx}
        sel = [i for i in sel if not (nodes[i]['tier'] == 1 and i in covered)]
    elif granularity == 'tier1':        # collapse everything to its Tier 1 bucket
        roll = {}
        for i in sel:
            t1 = i if nodes[i]['tier'] == 1 else id2idx.get(nodes[i]['parent'], i)
            roll[t1] = max(roll.get(t1, -1.0), float(agg[i]))
        for k, v in roll.items():
            agg[k] = v
        sel = sorted(roll, key=lambda k: -roll[k])
    # 'all' -> keep both Tier 1 and Tier 2

    sel = sel[:top_k]
    return [{'category': nodes[i]['enriched'], 'tier': nodes[i]['tier'],
             'score': round(float(agg[i]), 3)} for i in sel]

# quick sanity check
for r in classify("The Lakers beat the Celtics in last night's NBA basketball game."):
    print(f"{r['score']:.3f}  [T{r['tier']}]  {r['category']}")

0.814  [T2]  Sports > Basketball


### Step 4 \u2014 Test dataset (100 real pages: 50 EN + 50 IT)

Hardcoded DataFrame with `url`, `content`, `language`. The URLs are stable, navigable **Wikipedia articles** for manual validation; `content` is a clean topic-faithful excerpt. With the taxonomy capped at Tier 2, expect coarse tags.

In [5]:
import json, pandas as pd

_data = json.loads(r'''[{"url": "https://en.wikipedia.org/wiki/Basketball", "content": "Basketball is a team sport in which two teams of five players each try to score by shooting a ball through a hoop ten feet high. The game is played on a rectangular court and is governed internationally by FIBA, with the NBA being the premier professional league. Players advance the ball by dribbling and passing.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Pallacanestro", "content": "La pallacanestro è uno sport di squadra in cui due squadre di cinque giocatori cercano di segnare lanciando la palla in un canestro posto a poco più di tre metri di altezza. Si gioca su un campo rettangolare ed è regolata a livello internazionale dalla FIBA. L'NBA è il principale campionato professionistico.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Coffee", "content": "Coffee is a brewed beverage prepared from roasted coffee beans, the seeds of berries from the Coffea plant. It is one of the most popular drinks in the world, valued for its stimulating caffeine content. Major producing regions include Brazil, Vietnam and Colombia.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Caffè", "content": "Il caffè è una bevanda ottenuta dai semi tostati e macinati della pianta di Coffea. È tra le bevande più diffuse al mondo, apprezzata per il suo contenuto di caffeina dall'effetto stimolante. I principali paesi produttori sono Brasile, Vietnam e Colombia.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Pizza", "content": "Pizza is an Italian dish consisting of a flat base of leavened wheat dough topped with tomatoes, cheese and various other ingredients, then baked at a high temperature. Originating in Naples, it has become one of the most popular foods worldwide. The Margherita is a classic variety.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Pizza", "content": "La pizza è un piatto italiano costituito da un impasto di farina lievitato, condito con pomodoro, mozzarella e altri ingredienti, e cotto ad alta temperatura. Nata a Napoli, è diventata uno degli alimenti più diffusi al mondo. La Margherita è una delle varietà più classiche.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Tennis", "content": "Tennis is a racket sport played individually against a single opponent or between two teams of two players each. Players use a racket to hit a felt-covered ball over a net into the opponent's court. The four Grand Slam tournaments are the most prestigious events.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Tennis", "content": "Il tennis è uno sport con la racchetta che si gioca tra due giocatori o tra due coppie. I giocatori colpiscono con la racchetta una pallina rivestita di feltro facendola passare oltre la rete nel campo avversario. I tornei del Grande Slam sono gli eventi più prestigiosi.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Cycling", "content": "Cycling is the use of bicycles for transport, recreation, exercise or sport. Competitive road cycling includes events such as the Tour de France, while mountain biking and track cycling are other popular disciplines. It is valued as a low-impact aerobic exercise.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Ciclismo", "content": "Il ciclismo è l'uso della bicicletta per trasporto, svago o attività sportiva. Il ciclismo su strada comprende competizioni come il Tour de France e il Giro d'Italia, mentre la mountain bike e il ciclismo su pista sono altre discipline diffuse. È apprezzato come esercizio aerobico.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Swimming_(sport)", "content": "Swimming is an individual or team sport that involves moving through water using the limbs. Competitive swimming features strokes such as freestyle, backstroke, breaststroke and butterfly, and is a major Olympic discipline. It is also a popular recreational activity and form of exercise.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Nuoto", "content": "Il nuoto è uno sport individuale o di squadra che consiste nel muoversi nell'acqua usando gli arti. Il nuoto agonistico comprende stili come lo stile libero, il dorso, la rana e il delfino ed è una disciplina olimpica importante. È anche una diffusa attività ricreativa.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Golf", "content": "Golf is a club-and-ball sport in which players use various clubs to hit a ball into a series of holes on a course using as few strokes as possible. Major championships include the Masters and the Open. It is played on large outdoor courses with greens and fairways.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Golf", "content": "Il golf è uno sport in cui i giocatori usano delle mazze per mandare una pallina in una serie di buche con il minor numero di colpi possibile. Tra i tornei più importanti vi sono il Masters e l'Open. Si gioca su ampi percorsi all'aperto con green e fairway.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Marathon", "content": "The marathon is a long-distance running race with an official distance of 42.195 kilometres, usually run as a road race. It commemorates the legendary run of a Greek messenger. Major city marathons such as Boston, London and New York attract tens of thousands of runners.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Maratona", "content": "La maratona è una corsa di resistenza su strada con una distanza ufficiale di 42,195 chilometri. Trae origine dalla leggenda del messaggero greco che corse da Maratona ad Atene. Le grandi maratone cittadine come Boston, Londra e New York attirano decine di migliaia di partecipanti.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Guitar", "content": "The guitar is a fretted string instrument, typically with six strings, played by strumming or plucking. It exists in acoustic and electric forms and is central to many genres including rock, blues, flamenco and classical music. Sound is produced by vibrating strings over a resonating body.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Chitarra", "content": "La chitarra è uno strumento musicale a corde, generalmente sei, suonato pizzicando o strimpellando le corde. Esiste in versione acustica ed elettrica ed è centrale in molti generi come rock, blues, flamenco e musica classica. Il suono nasce dalla vibrazione delle corde su una cassa armonica.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Jazz", "content": "Jazz is a music genre that originated in the African-American communities of New Orleans in the late 19th and early 20th centuries. It is characterized by swing, blue notes, complex harmonies and improvisation. Notable figures include Louis Armstrong, Duke Ellington and Miles Davis.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Jazz", "content": "Il jazz è un genere musicale nato nelle comunità afroamericane di New Orleans tra la fine dell'Ottocento e l'inizio del Novecento. È caratterizzato dallo swing, dalle blue note, da armonie complesse e dall'improvvisazione. Tra i suoi protagonisti vi sono Louis Armstrong e Miles Davis.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Photography", "content": "Photography is the art and practice of creating images by recording light, typically with a camera. It is used for art, journalism, science and personal documentation. Digital sensors have largely replaced film, and composition, exposure and focus are key technical elements.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Fotografia", "content": "La fotografia è l'arte e la tecnica di creare immagini registrando la luce, di solito tramite una fotocamera. È usata nell'arte, nel giornalismo, nella scienza e nella documentazione personale. I sensori digitali hanno in gran parte sostituito la pellicola; composizione, esposizione e messa a fuoco sono elementi tecnici fondamentali.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Gardening", "content": "Gardening is the practice of growing and cultivating plants as part of horticulture, including ornamental flowers, vegetables and herbs. It can be done for food, decoration or relaxation. Activities include planting, watering, pruning and managing soil and pests.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Giardinaggio", "content": "Il giardinaggio è la pratica di coltivare piante, tra cui fiori ornamentali, ortaggi ed erbe aromatiche. Può essere svolto per produrre cibo, per decorazione o come hobby rilassante. Comprende attività come semina, irrigazione, potatura e cura del terreno.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Dog", "content": "The dog is a domesticated descendant of the wolf and one of the most common pets worldwide. Dogs have been bred into hundreds of breeds varying in size, appearance and temperament. They are kept as companions, working animals and for assistance and protection.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Cane", "content": "Il cane è un mammifero domestico discendente del lupo ed è uno degli animali da compagnia più diffusi al mondo. Esistono centinaia di razze che differiscono per taglia, aspetto e carattere. È tenuto come compagno, animale da lavoro e per assistenza e protezione.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Cat", "content": "The cat is a small domesticated carnivorous mammal and a popular household pet valued for companionship and its ability to hunt rodents. Cats communicate through vocalizations and body language and are known for their independence. Many breeds exist with varied coat patterns.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Gatto", "content": "Il gatto è un piccolo mammifero carnivoro domestico, uno degli animali da compagnia più amati, apprezzato per la compagnia e per la capacità di cacciare i roditori. Comunica con vocalizzi e linguaggio del corpo ed è noto per la sua indipendenza. Ne esistono molte razze.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Wine", "content": "Wine is an alcoholic drink made from fermented grapes. Different varieties of grapes and strains of yeasts produce different styles of wine, such as red, white and sparkling. Major wine-producing countries include Italy, France and Spain, and tasting evaluates aroma, body and tannins.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Vino", "content": "Il vino è una bevanda alcolica ottenuta dalla fermentazione dell'uva. Vitigni e lieviti diversi producono stili differenti, come vini rossi, bianchi e spumanti. Tra i maggiori produttori vi sono Italia, Francia e Spagna; la degustazione valuta aroma, corpo e tannini.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Beer", "content": "Beer is one of the oldest and most widely consumed alcoholic drinks, produced by brewing and fermenting cereal grains, most commonly malted barley. It is flavored with hops and styles include lager, ale, stout and IPA. Brewing is both an industrial process and a craft tradition.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Birra", "content": "La birra è una delle bevande alcoliche più antiche e diffuse, prodotta dalla fermentazione di cereali, in genere orzo maltato. È aromatizzata con il luppolo e comprende stili come lager, ale, stout e IPA. La produzione è sia industriale sia artigianale.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Sushi", "content": "Sushi is a Japanese dish of vinegared rice combined with various ingredients such as raw fish, seafood and vegetables. Common forms include nigiri, maki rolls and sashimi. It is typically served with soy sauce, wasabi and pickled ginger.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Sushi", "content": "Il sushi è un piatto giapponese a base di riso condito con aceto, abbinato a vari ingredienti come pesce crudo, frutti di mare e verdure. Tra le forme più comuni vi sono nigiri, maki e sashimi. Si serve di solito con salsa di soia, wasabi e zenzero marinato.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Smartphone", "content": "A smartphone is a portable device that combines mobile telephone and computing functions into one unit. Modern smartphones run apps, access the internet, take photos and include touchscreens and powerful processors. Leading platforms are Apple's iOS and Google's Android.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Smartphone", "content": "Lo smartphone è un dispositivo portatile che combina le funzioni di telefono cellulare e di computer. Gli smartphone moderni eseguono applicazioni, accedono a internet, scattano foto e dispongono di schermi touch e processori potenti. Le principali piattaforme sono iOS di Apple e Android di Google.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Video_game", "content": "A video game is an electronic game that involves interaction with a user interface to generate visual feedback on a display. Games span genres such as action, role-playing, strategy and sports, played on consoles, PCs and mobile devices. The industry is a major part of entertainment media.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Videogioco", "content": "Un videogioco è un gioco elettronico che prevede l'interazione con un'interfaccia per generare un riscontro visivo su uno schermo. I generi spaziano da azione, giochi di ruolo, strategia e sport, giocati su console, PC e dispositivi mobili. L'industria è un settore importante dell'intrattenimento.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Bitcoin", "content": "Bitcoin is a decentralized digital currency, created in 2009, that operates without a central bank or single administrator. Transactions are verified by network nodes through cryptography and recorded on a public ledger called a blockchain. New bitcoins are created through a process called mining.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Bitcoin", "content": "Bitcoin è una valuta digitale decentralizzata, creata nel 2009, che funziona senza una banca centrale o un amministratore unico. Le transazioni sono verificate dai nodi della rete tramite la crittografia e registrate su un registro pubblico chiamato blockchain. I nuovi bitcoin nascono tramite un processo detto mining.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Cryptocurrency", "content": "A cryptocurrency is a digital or virtual currency secured by cryptography, making it nearly impossible to counterfeit. Most operate on decentralized networks based on blockchain technology. Besides Bitcoin, thousands of alternative coins exist, and the market is known for high volatility.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Criptovaluta", "content": "Una criptovaluta è una valuta digitale protetta dalla crittografia, che la rende quasi impossibile da falsificare. La maggior parte funziona su reti decentralizzate basate sulla tecnologia blockchain. Oltre a Bitcoin esistono migliaia di valute alternative e il mercato è noto per l'elevata volatilità.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Yoga", "content": "Yoga is a group of physical, mental and spiritual practices originating in ancient India. Modern yoga commonly emphasizes physical postures, breathing techniques and meditation, and is widely practiced for fitness and stress relief. It has roots in Hindu philosophy.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Yoga", "content": "Lo yoga è un insieme di pratiche fisiche, mentali e spirituali nate nell'antica India. Lo yoga moderno enfatizza posizioni fisiche, tecniche di respirazione e meditazione, ed è ampiamente praticato per il benessere e per ridurre lo stress. Ha radici nella filosofia indiana.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Astronomy", "content": "Astronomy is the natural science that studies celestial objects and phenomena such as stars, planets, galaxies and the origins of the universe. It uses observation, mathematics and physics, and instruments like telescopes. It is one of the oldest sciences.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Astronomia", "content": "L'astronomia è la scienza naturale che studia i corpi e i fenomeni celesti come stelle, pianeti, galassie e le origini dell'universo. Si avvale di osservazione, matematica e fisica, e di strumenti come i telescopi. È una delle scienze più antiche.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/University", "content": "A university is an institution of higher education and research that grants academic degrees in various disciplines. Universities typically offer undergraduate and postgraduate programs and conduct scholarly research. They play a central role in education and the advancement of knowledge.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Università", "content": "Un'università è un'istituzione di istruzione superiore e ricerca che conferisce titoli accademici in varie discipline. Le università offrono corsi di laurea e post-laurea e svolgono attività di ricerca scientifica. Hanno un ruolo centrale nell'istruzione e nel progresso della conoscenza.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Fashion", "content": "Fashion is a form of self-expression and a popular aesthetic in clothing, footwear, accessories and style. The fashion industry encompasses design, manufacturing and retail, and trends are shaped by designers, runway shows and culture. Major fashion capitals include Paris, Milan and New York.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Moda", "content": "La moda è una forma di espressione personale e un'estetica diffusa che riguarda abbigliamento, calzature e accessori. L'industria della moda comprende design, produzione e vendita, e le tendenze sono influenzate da stilisti, sfilate e cultura. Tra le capitali della moda vi sono Parigi, Milano e New York.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Pregnancy", "content": "Pregnancy is the period during which one or more offspring develops inside a woman's uterus, typically lasting about forty weeks. It is divided into three trimesters and involves significant physical and hormonal changes. Prenatal care helps monitor the health of mother and baby.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Gravidanza", "content": "La gravidanza è il periodo durante il quale uno o più feti si sviluppano nell'utero della donna, della durata di circa quaranta settimane. È suddivisa in tre trimestri e comporta importanti cambiamenti fisici e ormonali. Le visite prenatali aiutano a monitorare la salute di madre e bambino.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Diabetes", "content": "Diabetes is a chronic metabolic disease characterized by elevated levels of blood glucose, resulting from problems with insulin production or action. The main types are type 1 and type 2. Management involves diet, exercise, monitoring blood sugar and sometimes medication or insulin.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Diabete_mellito", "content": "Il diabete mellito è una malattia metabolica cronica caratterizzata da elevati livelli di glucosio nel sangue, dovuti a problemi nella produzione o nell'azione dell'insulina. I tipi principali sono il tipo 1 e il tipo 2. La gestione prevede dieta, attività fisica, monitoraggio della glicemia ed eventuali farmaci.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Mediterranean_diet", "content": "The Mediterranean diet is a way of eating based on the traditional cuisines of countries bordering the Mediterranean Sea. It emphasizes vegetables, fruits, whole grains, olive oil, fish and moderate wine, with limited red meat. It is widely associated with cardiovascular health benefits.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Dieta_mediterranea", "content": "La dieta mediterranea è un modello alimentare basato sulle cucine tradizionali dei paesi del Mediterraneo. Privilegia verdura, frutta, cereali integrali, olio d'oliva e pesce, con un consumo limitato di carne rossa. È ampiamente associata a benefici per la salute cardiovascolare.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Artificial_intelligence", "content": "Artificial intelligence is the field of computer science focused on creating systems capable of performing tasks that typically require human intelligence, such as reasoning, learning and perception. Modern AI relies heavily on machine learning and neural networks. Applications range from language models to robotics.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Intelligenza_artificiale", "content": "L'intelligenza artificiale è il campo dell'informatica che si occupa di creare sistemi capaci di svolgere compiti che normalmente richiedono l'intelligenza umana, come ragionamento, apprendimento e percezione. L'IA moderna si basa molto sull'apprendimento automatico e sulle reti neurali. Le applicazioni vanno dai modelli linguistici alla robotica.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Machine_learning", "content": "Machine learning is a branch of artificial intelligence in which algorithms learn patterns from data to make predictions or decisions without being explicitly programmed. Common approaches include supervised, unsupervised and reinforcement learning. It powers applications such as recommendation systems and image recognition.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Apprendimento_automatico", "content": "L'apprendimento automatico è un ramo dell'intelligenza artificiale in cui gli algoritmi apprendono schemi dai dati per fare previsioni o decisioni senza essere programmati esplicitamente. Gli approcci comuni includono l'apprendimento supervisionato, non supervisionato e per rinforzo. Alimenta applicazioni come i sistemi di raccomandazione e il riconoscimento di immagini.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Electric_vehicle", "content": "An electric vehicle is a vehicle that uses one or more electric motors for propulsion, powered by rechargeable batteries instead of an internal combustion engine. EVs produce no tailpipe emissions and are central to efforts to reduce transport-related carbon. Charging infrastructure and battery range are key considerations.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Veicolo_elettrico", "content": "Un veicolo elettrico è un veicolo che utilizza uno o più motori elettrici per la propulsione, alimentati da batterie ricaricabili anziché da un motore a combustione interna. I veicoli elettrici non producono emissioni allo scarico e sono centrali nella riduzione delle emissioni dei trasporti. Autonomia e infrastrutture di ricarica sono aspetti chiave.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Tourism", "content": "Tourism is travel for pleasure or business and the commercial activity of providing services for tourists. It is a major economic sector for many countries, encompassing accommodation, transport, attractions and hospitality. Popular forms include cultural, beach and adventure tourism.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Turismo", "content": "Il turismo è il viaggiare per piacere o per affari e l'attività economica di fornire servizi ai turisti. È un settore economico importante per molti paesi e comprende alloggio, trasporti, attrazioni e ospitalità. Tra le sue forme vi sono il turismo culturale, balneare e d'avventura.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Rome", "content": "Rome is the capital city of Italy and a major historical and cultural center. Founded according to tradition in 753 BC, it was the heart of the Roman Empire and is home to landmarks such as the Colosseum, the Roman Forum and Vatican City. It is a leading tourist destination.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Roma", "content": "Roma è la capitale d'Italia e un importante centro storico e culturale. Fondata secondo la tradizione nel 753 a.C., fu il cuore dell'Impero romano e ospita monumenti come il Colosseo, il Foro Romano e la Città del Vaticano. È una delle principali mete turistiche del mondo.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Paris", "content": "Paris is the capital of France and one of the world's leading centers of art, fashion and culture. Situated on the river Seine, it is known for landmarks such as the Eiffel Tower, the Louvre and Notre-Dame. It is among the most visited cities in the world.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Parigi", "content": "Parigi è la capitale della Francia e uno dei principali centri mondiali di arte, moda e cultura. Situata sulla Senna, è celebre per monumenti come la Torre Eiffel, il Louvre e Notre-Dame. È tra le città più visitate al mondo.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Painting", "content": "Painting is the practice of applying pigment to a surface to create an image, composition or expression. It is one of the oldest art forms and includes styles such as realism, impressionism and abstraction. Mediums include oil, watercolor, acrylic and fresco.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Pittura", "content": "La pittura è la pratica di applicare pigmenti su una superficie per creare un'immagine o un'espressione artistica. È una delle forme d'arte più antiche e comprende stili come il realismo, l'impressionismo e l'astrattismo. Tra le tecniche vi sono olio, acquerello, acrilico e affresco.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Chess", "content": "Chess is a two-player strategy board game played on a checkered board with 64 squares. Each player controls sixteen pieces with different movements, the goal being to checkmate the opponent's king. It is governed internationally by FIDE and has a long competitive tradition.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Scacchi", "content": "Gli scacchi sono un gioco di strategia per due giocatori che si svolge su una scacchiera di 64 caselle. Ogni giocatore controlla sedici pezzi con movimenti diversi e l'obiettivo è dare scacco matto al re avversario. Il gioco è regolato a livello internazionale dalla FIDE.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Whisky", "content": "Whisky is a distilled alcoholic beverage made from fermented grain mash, aged in wooden casks. Varieties include Scotch, Irish, bourbon and rye, distinguished by ingredients and production methods. Aging in oak gives whisky its color and much of its flavor.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Whisky", "content": "Il whisky è una bevanda alcolica distillata ottenuta da un mosto di cereali fermentati e invecchiata in botti di legno. Le varietà comprendono Scotch, irlandese, bourbon e rye, che si distinguono per ingredienti e metodi di produzione. L'invecchiamento in rovere ne determina colore e aroma.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Italian_cuisine", "content": "Italian cuisine is a Mediterranean culinary tradition known for its regional diversity, simple high-quality ingredients and dishes such as pasta, pizza and risotto. It emphasizes ingredients like olive oil, tomatoes, cheese and fresh herbs. It is one of the most popular cuisines in the world.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Cucina_italiana", "content": "La cucina italiana è una tradizione culinaria mediterranea nota per la varietà regionale, gli ingredienti semplici e di qualità e piatti come pasta, pizza e risotto. Dà grande importanza a ingredienti come olio d'oliva, pomodoro, formaggi ed erbe fresche. È tra le cucine più amate al mondo.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Vegetarianism", "content": "Vegetarianism is the practice of abstaining from eating meat and may also include avoiding other animal products. People adopt it for ethical, health, environmental or religious reasons. Variants range from lacto-ovo vegetarianism to stricter plant-based diets.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Vegetarianismo", "content": "Il vegetarianismo è la pratica di astenersi dal consumo di carne e può includere anche l'esclusione di altri prodotti animali. Viene adottato per motivi etici, di salute, ambientali o religiosi. Le varianti vanno dalla dieta latto-ovo-vegetariana a regimi più rigidi a base vegetale.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Solar_energy", "content": "Solar energy is radiant light and heat from the Sun harnessed using technologies such as photovoltaic panels and solar thermal collectors. It is a renewable energy source used to generate electricity and heat. It plays a growing role in reducing dependence on fossil fuels.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Energia_solare", "content": "L'energia solare è la luce e il calore irradiati dal Sole, sfruttati tramite tecnologie come i pannelli fotovoltaici e i collettori solari termici. È una fonte di energia rinnovabile usata per produrre elettricità e calore. Ha un ruolo crescente nella riduzione della dipendenza dai combustibili fossili.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Renewable_energy", "content": "Renewable energy is energy from sources that are naturally replenished, such as sunlight, wind, water and geothermal heat. It is central to efforts to reduce greenhouse gas emissions and combat climate change. Wind and solar power have grown rapidly in recent years.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Energia_rinnovabile", "content": "L'energia rinnovabile è energia proveniente da fonti che si rigenerano naturalmente, come luce solare, vento, acqua e calore geotermico. È centrale negli sforzi per ridurre le emissioni di gas serra e contrastare il cambiamento climatico. Eolico e solare sono cresciuti rapidamente negli ultimi anni.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Renaissance", "content": "The Renaissance was a period of European cultural, artistic and intellectual revival spanning roughly the 14th to the 17th century, beginning in Italy. It marked a renewed interest in classical antiquity and produced artists such as Leonardo da Vinci and Michelangelo. It influenced art, science and philosophy.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Rinascimento", "content": "Il Rinascimento fu un periodo di rinascita culturale, artistica e intellettuale europea che si estese all'incirca dal XIV al XVII secolo, con origine in Italia. Segnò un rinnovato interesse per l'antichità classica e produsse artisti come Leonardo da Vinci e Michelangelo. Influenzò arte, scienza e filosofia.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Mortgage_loan", "content": "A mortgage is a loan used to purchase real estate, in which the property serves as collateral for the lender. Borrowers repay the principal plus interest over a set term, often decades. Interest rates may be fixed or variable, and failure to repay can lead to foreclosure.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Mutuo", "content": "Il mutuo è un prestito utilizzato per acquistare un immobile, in cui la proprietà funge da garanzia per la banca. Il debitore restituisce il capitale più gli interessi in un periodo prestabilito, spesso di molti anni. Il tasso può essere fisso o variabile e il mancato pagamento può portare al pignoramento.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Stock_market", "content": "A stock market is a public marketplace where shares of companies are issued, bought and sold. It enables companies to raise capital and investors to own a stake in businesses. Prices fluctuate based on supply, demand and economic conditions, tracked by indices such as the S&P 500.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Borsa_valori", "content": "La borsa valori è un mercato pubblico in cui le azioni delle società vengono emesse, comprate e vendute. Permette alle aziende di raccogliere capitale e agli investitori di possedere quote delle imprese. I prezzi variano in base a domanda, offerta e condizioni economiche, monitorati da indici come l'S&P 500.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Personal_finance", "content": "Personal finance refers to managing an individual's or household's money, including budgeting, saving, investing and planning for retirement. It also covers managing debt, insurance and taxes. Sound personal finance aims to achieve financial security and meet life goals.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Finanza_personale", "content": "La finanza personale riguarda la gestione del denaro di un individuo o di una famiglia, comprendendo budget, risparmio, investimenti e pianificazione della pensione. Include anche la gestione di debiti, assicurazioni e imposte. Una buona finanza personale mira a raggiungere la sicurezza economica.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Climate_change", "content": "Climate change refers to long-term shifts in temperatures and weather patterns, largely driven since the 1800s by human activities such as burning fossil fuels. It leads to rising global temperatures, melting ice and more extreme weather. Mitigation focuses on reducing greenhouse gas emissions.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Cambiamenti_climatici", "content": "Il cambiamento climatico indica variazioni a lungo termine delle temperature e dei modelli meteorologici, causate in gran parte dal 1800 dalle attività umane come la combustione di combustibili fossili. Provoca aumento delle temperature globali, scioglimento dei ghiacci ed eventi estremi. La mitigazione punta a ridurre i gas serra.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Association_football", "content": "Association football, commonly known as soccer or football, is a team sport played between two teams of eleven players using a spherical ball. It is the world's most popular sport, governed by FIFA, with the World Cup as its premier tournament. Players score by getting the ball into the opposing goal.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Calcio_(sport)", "content": "Il calcio è uno sport di squadra giocato tra due squadre di undici giocatori con un pallone sferico. È lo sport più popolare al mondo, regolato dalla FIFA, e ha nel Campionato del Mondo il suo torneo più importante. Si segna facendo entrare il pallone nella porta avversaria.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Alpine_skiing", "content": "Alpine skiing is the sport of sliding down snow-covered slopes on skis with fixed-heel bindings. Competitive disciplines include slalom, giant slalom, super-G and downhill. It is a popular winter recreational activity and an Olympic sport.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Sci_alpino", "content": "Lo sci alpino è lo sport che consiste nel discendere pendii innevati con gli sci dotati di attacchi a tallone fisso. Le discipline agonistiche comprendono slalom, slalom gigante, supergigante e discesa libera. È un diffuso sport invernale ricreativo e disciplina olimpica.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Opera", "content": "Opera is a form of theatre in which music is a fundamental component and dramatic roles are sung. It originated in Italy at the end of the 16th century and combines singing, orchestral music, acting and stagecraft. Famous composers include Verdi, Puccini and Mozart.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Opera_lirica", "content": "L'opera lirica è una forma di teatro in cui la musica è una componente fondamentale e i ruoli drammatici sono cantati. Nata in Italia alla fine del Cinquecento, unisce canto, musica orchestrale, recitazione e scenografia. Tra i compositori celebri vi sono Verdi, Puccini e Mozart.", "language": "it"}, {"url": "https://en.wikipedia.org/wiki/Heavy_metal_music", "content": "Heavy metal is a genre of rock music that developed in the late 1960s and early 1970s, characterized by amplified distortion, extended guitar solos and emphatic beats. Pioneering bands include Black Sabbath, Led Zeppelin and Deep Purple. It has spawned many subgenres.", "language": "en"}, {"url": "https://it.wikipedia.org/wiki/Heavy_metal", "content": "L'heavy metal è un genere di musica rock sviluppatosi tra la fine degli anni Sessanta e i primi anni Settanta, caratterizzato da distorsione amplificata, lunghi assoli di chitarra e ritmi marcati. Tra i gruppi pionieri vi sono Black Sabbath, Led Zeppelin e Deep Purple. Ha generato molti sottogeneri.", "language": "it"}]''')
test_df = pd.DataFrame(_data)[['url', 'content', 'language']]
print(len(test_df), 'examples |', test_df['language'].value_counts().to_dict())
pd.set_option('display.max_colwidth', 80)
test_df.head(6)

100 examples | {'en': 50, 'it': 50}


,url,content,language
0,https://en.wikipedia.org/wiki/Basketball,Basketball is a team sport in which two teams of five players each try to sc...,en
1,https://it.wikipedia.org/wiki/Pallacanestro,La pallacanestro è uno sport di squadra in cui due squadre di cinque giocato...,it
2,https://en.wikipedia.org/wiki/Coffee,"Coffee is a brewed beverage prepared from roasted coffee beans, the seeds of...",en
3,https://it.wikipedia.org/wiki/Caffè,Il caffè è una bevanda ottenuta dai semi tostati e macinati della pianta di ...,it
4,https://en.wikipedia.org/wiki/Pizza,Pizza is an Italian dish consisting of a flat base of leavened wheat dough t...,en
5,https://it.wikipedia.org/wiki/Pizza,"La pizza è un piatto italiano costituito da un impasto di farina lievitato, ...",it


### Step 5 \u2014 Run and inspect

`predict_df` classifies every page and shows the navigable URL next to predicted Tier 1\u20132 categories and cosine scores. The last cell drills into one example \u2014 use it to tune `floor / margin / top_k`.

In [6]:
from IPython.display import display

def predict_df(df, **kw):
    out = []
    for _, row in df.iterrows():
        preds = classify(row['content'], **kw)
        out.append({
            'language': row['language'],
            'url': row['url'],
            'predictions': " ; ".join(f"{p['category']} ({p['score']})" for p in preds) or "(none)",
        })
    return pd.DataFrame(out)

results = predict_df(test_df, floor=0.40, margin=0.10, top_k=3, granularity='specific')

pd.set_option('display.max_colwidth', 200)   # keep the HTML table light in Colab
pd.set_option('display.max_rows', 100)
display(results)

,language,url,predictions
0,en,https://en.wikipedia.org/wiki/Basketball,Sports > Basketball (0.809) ; Sports > College Sports (0.726) ; Sports > Sports Equipment (0.722)
1,it,https://it.wikipedia.org/wiki/Pallacanestro,Sports > Basketball (0.812)
2,en,https://en.wikipedia.org/wiki/Coffee,Food & Drink > Non-Alcoholic Beverages (0.727) ; Food & Drink > World Cuisines (0.723)
3,it,https://it.wikipedia.org/wiki/Caffè,Food & Drink > World Cuisines (0.729) ; Food & Drink > Non-Alcoholic Beverages (0.713)
4,en,https://en.wikipedia.org/wiki/Pizza,Food & Drink > World Cuisines (0.791) ; Food & Drink > Cooking (0.723) ; Food & Drink > Food Movements (0.696)
5,it,https://it.wikipedia.org/wiki/Pizza,Food & Drink > World Cuisines (0.754) ; Food & Drink > Cooking (0.667) ; Food & Drink > Food Movements (0.656)
6,en,https://en.wikipedia.org/wiki/Tennis,Sports > Tennis (0.81) ; Sports > Table Tennis (0.74)
7,it,https://it.wikipedia.org/wiki/Tennis,Sports > Tennis (0.832)
8,en,https://en.wikipedia.org/wiki/Cycling,Sports > Cycling (0.77) ; Sports > Extreme Sports (0.701) ; Healthy Living > Fitness and Exercise (0.682)
9,it,https://it.wikipedia.org/wiki/Ciclismo,Sports > Cycling (0.786) ; Sports > Gymnastics (0.73) ; Sports > Extreme Sports (0.729)


In [7]:
# Inspect a single example (change the index to explore)
i = 0
print("URL :", test_df.iloc[i]['url'])
print("LANG:", test_df.iloc[i]['language'])
print("TEXT:", test_df.iloc[i]['content'][:300], "...\n")
for p in classify(test_df.iloc[i]['content'], top_k=6):
    print(f"  {p['score']:.3f}  [T{p['tier']}]  {p['category']}")

URL : https://en.wikipedia.org/wiki/Basketball
LANG: en
TEXT: Basketball is a team sport in which two teams of five players each try to score by shooting a ball through a hoop ten feet high. The game is played on a rectangular court and is governed internationally by FIBA, with the NBA being the premier professional league. Players advance the ball by dribblin ...

  0.809  [T2]  Sports > Basketball
  0.726  [T2]  Sports > College Sports
  0.722  [T2]  Sports > Sports Equipment


## Operational notes

- **Instruction matters.** Qwen3 is trained to follow the query instruction; editing `INSTRUCT` to describe your real task can shift results a few points. Keep the exact `Instruct: ...\nQuery: ` format, and apply it only to pages, never to labels.
- **Threshold tuning (cosine scale).** No reranker here, so `floor`/`margin` act on cosine again. Calibrate on the 100 annotated examples with a small grid search.
- **Granularity.** `'specific'` for Tier 2 where confident, `'tier1'` for the broadest, most stable buckets.
- **Memory on T4.** fp16 keeps the 4B model around 8 GB. If you OOM, lower `batch_size`, reduce `max_seq_length`, or switch `MODEL_NAME` to `Qwen/Qwen3-Embedding-0.6B`.
- **Matryoshka (optional).** Qwen3 supports truncating the embedding dimension to cut vector storage with minor quality loss \u2014 irrelevant for 360 labels, useful only at large index scale.
- **Real pages.** Excerpts here are short; for long raw pages raise `max_seq_length`.
- **Persistence.** Compute `label_emb` once and save it (`np.save`); per page the cost is a single instructed encoding plus one matrix-vector product.